### Моделирование БИНС карьерного самосвала БелАЗ-7513

In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib qt5

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORT_DIR = PROJECT_ROOT / 'reports'
DATA_DIR   = PROJECT_ROOT / 'generated_data'
MOTION_DIR   = PROJECT_ROOT / 'motion_profiles'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'REPORT_DIR: {REPORT_DIR}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'MOTION_DIR: {MOTION_DIR}')

REPORT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
MOTION_DIR.mkdir(exist_ok=True)

# Config libs
from src.config.config import IMU_ERR, GPS_ERR
from src.config.constants import FS_IMU, FS_GPS, INIT_LAT, INIT_LON, INIT_ALT, A_WGS84


Генерация всех профилей движения

In [ ]:
from src.simulation.motion_generator import generate_all

generate_all(out_dir=MOTION_DIR)

Генерируем сценарий -> ИНС, ГНСС

In [ ]:
from src.analysis.frames import prepare_scenario

# Сценарии: 01_stationary, 02_acceleration, 03_straight_cruise,
#           04_braking, 05_turn, 06_uphill, 07_downhill_turn,
#           10_accel_cruise_brake, 99_full_mission

# S = prepare_scenario('01_stationary', MOTION_DIR, DATA_DIR, vibration_mode='harmonics_plus_noise')
# S = prepare_scenario('02_acceleration', MOTION_DIR, DATA_DIR, vibration_mode='harmonics_plus_noise')
S = prepare_scenario('03_straight_cruise', MOTION_DIR, DATA_DIR, vibration_mode='harmonics_plus_noise')
# S = prepare_scenario('05_turn', MOTION_DIR, DATA_DIR, vibration_mode='harmonics_plus_noise')
# S = prepare_scenario('10_accel_cruise_brake', MOTION_DIR, DATA_DIR, vibration_mode='harmonics_plus_noise')

imu_time = S['imu_time']
accel = S['accel']
gyro = S['gyro']
ref_accel = S['ref_accel']
ref_gyro = S['ref_gyro']
accel_vib = S['accel_vib']
gyro_vib = S['gyro_vib']

ref_pos_ned   = S['ref_pos_ned']
ref_vel_ned   = S['ref_vel_ned']
ref_euler_ned = S['ref_euler_ned']
ref_euler_deg = np.rad2deg(S['ref_euler_ned'])

gps_pos_ned=S['gps_pos_ned']
gps_vel_ned=S['gps_vel_ned']
gps_idx=S['gps_idx'] # - на какие индексы ИИМ приходиться GPS
q0=S['q0']
lat0_deg=S['lat0_deg']
lat0_rad=np.deg2rad(S['lat0_deg'])
noise=S['noise']

In [ ]:
from src.analysis.visualization import plot_accel_signal, plot_gyro_signal

_ = plot_accel_signal(imu_time, accel, ' (без вибраций)', save_dir=f'{REPORT_DIR}/raw')
_ = plot_gyro_signal(imu_time, gyro, ' (без вибраций)', save_dir=f'{REPORT_DIR}/raw')

# _ = plot_accel_signal(imu_time, accel_vib, ' (с вибрациями ДВС)',
#                   save_dir=f'{REPORT_DIR}/with_vib')
# _ = plot_gyro_signal(imu_time, gyro_vib, ' (с вибрациями ДВС)',
#                  save_dir=f'{REPORT_DIR}/with_vib')

In [ ]:
from src.analysis.visualization import plot_psd, plot_spectrogram_2d, plot_spectrogram_3d

# plot_psd(imu_time, accel, fs=400,
#          sensor_name='Акселерометр', units='м/с²',
#          save_dir=f'{REPORT_DIR}/raw', fname='psd_accel')
# _ = plot_psd(imu_time, gyro, fs=400,
#          sensor_name='Гироскоп', units='рад/с',
#          save_dir=f'{REPORT_DIR}/raw', fname='psd_gyro')

plot_psd(imu_time, accel_vib, fs=400,
         sensor_name='Акселерометр', units='м/с²',
         save_dir=REPORT_DIR, fname='psd_accel')
_ = plot_psd(imu_time, gyro_vib, fs=400,
         sensor_name='Гироскоп', units='рад/с',
         save_dir=REPORT_DIR, fname='psd_gyro')

# Спектрограмма Z-оси акселерометра (там вибрация самая сильная)
# _ = plot_spectrogram_2d(imu_time, accel_vib[:, 2], fs=400,
#                     channel_name='Z', sensor_name='Акселерометр',
#                     save_dir=REPORT_DIR)
# _ = plot_spectrogram_3d(imu_time, accel_vib[:, 2], fs=400,
#                     channel_name='Z', sensor_name='акселерометра',
#                     save_dir=REPORT_DIR)

Девиация Аллана

In [ ]:
import os, tempfile
import numpy as np
from gnss_ins_sim.sim import imu_model, ins_sim
from src.config.config import IMU_ERR
from src.analysis.visualization import plot_allan_simulated

# --- Параметры ---
FS_ALLAN = 100.0          # Гц (100 достаточно и быстрее; для ВЧ-эффектов можно 400)
DURATION_H = 3.0          # длительность статики, часы
duration_s = int(DURATION_H * 3600)

# --- Генерация длинной статики (профиль НЕ сохраняем, пишем во временный файл) ---
HEADER = (
    "ini lat (deg),ini lon (deg),ini alt (m),"
    "ini vx_body (m/s),ini vy_body (m/s),ini vz_body (m/s),"
    "ini yaw (deg),ini pitch (deg),ini roll (deg)\n"
    "55.44,86.05,250,0,0,0,0,0,0\n"
    "command type,yaw (deg),pitch (deg),roll (deg),"
    "vx_body (m/s),vy_body (m/s),vz_body (m/s),"
    "command duration (s),GPS visibility\n"
)

tmp_dir = tempfile.mkdtemp()
motion_csv = os.path.join(tmp_dir, "allan_static.csv")

with open(motion_csv, "w") as f:
    f.write(HEADER)
    f.write(f"1,0,0,0,0,0,0,{duration_s},1\n")   # команда удержания состояния

# --- Прогон симулятора (только ИИМ, без GPS) ---
imu = imu_model.IMU(accuracy=IMU_ERR, axis=6, gps=False)
sim = ins_sim.Sim([FS_ALLAN, 0.0, 0.0], motion_csv,
                  ref_frame=1, imu=imu, mode=None, env=None, algorithm=None)
sim.run(1)

gyro_static = sim.dmgr.get_data_all('gyro').data[0]    # рад/с
accel_static = sim.dmgr.get_data_all('accel').data[0]  # м/с²
print(f"Статика: {len(gyro_static)} сэмплов = {len(gyro_static)/FS_ALLAN/3600:.2f} ч")

# --- Расчёт и отрисовка ---
fig_g, fig_a = plot_allan_simulated(gyro_static, accel_static, FS_ALLAN,
                                    save_dir=f'{REPORT_DIR}/allan')

Глава 1) Оценка ориентации

Фильтр Маджвика

In [ ]:
from src.navigation.sins import make_sins

sins_madg = make_sins('madgwick', fs=400.0, beta=0.03,
                      init_pos_ned=ref_pos_ned[0],
                      init_vel_ned=ref_vel_ned[0],
                      init_quat=q0, 
                      lat0_deg=lat0_deg
                     )

out_madg = sins_madg.run(accel_vib, gyro_vib)
# out_madg = sins_madg.run(accel, gyro)

In [ ]:
from src.analysis.visualization import plot_trajectory_3d, plot_velocity_ned, plot_euler_angles, plot_position_error, plot_velocity_error, plot_attitude_error

# Истинное и оценненое
_ = plot_trajectory_3d(ref_pos_ned, out_madg['p_n'], save_dir=f'{REPORT_DIR}/madgwick')
_ = plot_velocity_ned(imu_time, ref_vel_ned, out_madg['v_n'], save_dir=f'{REPORT_DIR}/madgwick')
_ = plot_euler_angles(imu_time, ref_euler_ned, out_madg['euler'], save_dir=f'{REPORT_DIR}/madgwick')

# Ошибки
_ = plot_position_error(imu_time, out_madg['p_n'], ref_pos_ned, save_dir=f'{REPORT_DIR}/madgwick')
_ = plot_velocity_error(imu_time, ref_vel_ned, out_madg['v_n'], save_dir=f'{REPORT_DIR}/madgwick')
_ = plot_attitude_error(imu_time, out_madg['euler'], ref_euler_ned, save_dir=f'{REPORT_DIR}/madgwick')

БИНС с фильтром Махони

In [ ]:
from src.navigation.sins import make_sins
from src.analysis.visualization import plot_trajectory_3d, plot_velocity_ned, plot_euler_angles, plot_position_error, plot_attitude_error

sins_mah = make_sins('mahony', fs=400.0, k_P=0.1, k_I=0.001,
                     init_pos_ned=ref_pos_ned[0],
                     init_vel_ned=ref_vel_ned[0],
                     init_quat=q0, 
                     lat0_deg=lat0_deg
                     )

# out_mah = sins_mah.run(accel_vib, gyro_vib)
out_mah = sins_mah.run(accel, gyro)

In [ ]:
_ = plot_trajectory_3d(ref_pos_ned, out_mah['p_n'], save_dir=f'{REPORT_DIR}/mahony')
_ = plot_velocity_ned(imu_time, ref_vel_ned, v_h, save_dir=f'{REPORT_DIR}/mahony')
_ = plot_euler_angles(imu_time, ref_euler_ned, eul_h, save_dir=f'{REPORT_DIR}/mahony')
_ = plot_position_error(imu_time, p_h, ref_pos_ned, save_dir=f'{REPORT_DIR}/mahony')
_ = plot_attitude_error(imu_time, eul_h, ref_euler_ned, save_dir=f'{REPORT_DIR}/mahony')

Анализ β для фильтра Маджвика

In [ ]:
from src.analysis.visualization import plot_beta_sweep

# beta_values = [0.005, 0.02, 0.05, 0.1, 0.3, 1.0]
beta_values = [1, 0.25, 0.1, 0.05, 0.02, 0.005]
beta_results = {}

for b in beta_values:
   
    sins_b = make_sins('madgwick', fs=400.0, beta=b,
                       init_pos_ned=ref_pos_ned[0],
                       init_vel_ned=ref_vel_ned[0],
                       init_quat=q0, 
                       lat0_deg=lat0_deg
                       )
    out_b = sins_b.run(accel_vib, gyro_vib)
    eul_b = out_b['euler']
    beta_results = eul_b
    print(f"  β = {b:.3f} готово")

In [ ]:
plot_beta_sweep(imu_time, beta_results, ref_euler_ned,
                save_dir=f'{REPORT_DIR}/beta_sweep')

Глава 2) Схемы комплексирования БИНС/ГНСС

EKF

In [ ]:
from src.analysis.run_methods import run_kalman

p_ekf, v_ekf, e_ekf = run_kalman(InsGnssEKF, S, accel, gyro)

In [ ]:
from src.analysis.visualization import plot_trajectory_3d, plot_velocity_ned, plot_euler_angles, plot_position_error, plot_attitude_error

_ = plot_trajectory_3d(ref_pos_ned, p_ekf, save_dir=f'{REPORT_DIR}/ekf')
_ = plot_velocity_ned(imu_time, ref_vel_ned, v_ekf, save_dir=f'{REPORT_DIR}/ekf')
_ = plot_euler_angles(imu_time, ref_euler_ned, e_ekf, save_dir=f'{REPORT_DIR}/ekf')
_ = plot_position_error(imu_time, p_ekf, ref_pos_ned, save_dir=f'{REPORT_DIR}/ekf')
_ = plot_attitude_error(imu_time, e_ekf, ref_euler_ned, save_dir=f'{REPORT_DIR}/ekf')

UKF

In [ ]:
from src.analysis.run_methods import run_kalman

p_ukf, v_ukf, e_ukf = run_kalman(InsGnssUKF, S, accel, gyro)

In [ ]:
from src.analysis.visualization import plot_trajectory_3d, plot_velocity_ned, plot_euler_angles, plot_position_error, plot_attitude_error

_ = plot_trajectory_3d(ref_pos_ned, p_ukf, save_dir=f'{REPORT_DIR}/ukf')
_ = plot_velocity_ned(imu_time, ref_vel_ned, v_ekf, save_dir=f'{REPORT_DIR}/ukf')
_ = plot_euler_angles(imu_time, ref_euler_ned, e_ukf, save_dir=f'{REPORT_DIR}/ukf')
_ = plot_position_error(imu_time, p_ekf, ref_pos_ned, save_dir=f'{REPORT_DIR}/ukf')
_ = plot_attitude_error(imu_time, e_ukf, ref_euler_ned, save_dir=f'{REPORT_DIR}/ukf')

FGO

In [ ]:
from src.analysis.run_methods import run_fgo

p_fgo, v_fgo, e_fgo = run_fgo(S, accel, gyro)

In [ ]:
from src.analysis.visualization import plot_trajectory_3d, plot_velocity_ned, plot_euler_angles, plot_position_error, plot_attitude_error

_ = plot_trajectory_3d(ref_pos_ned, p_fgo, save_dir=f'{REPORT_DIR}/fgo')
_ = plot_velocity_ned(imu_time, ref_vel_ned, v_fgo, save_dir=f'{REPORT_DIR}/fgo')
_ = plot_euler_angles(imu_time, ref_euler_ned, e_fgo, save_dir=f'{REPORT_DIR}/fgo')
_ = plot_position_error(imu_time, p_fgo, ref_pos_ned, save_dir=f'{REPORT_DIR}/fgo')
_ = plot_attitude_error(imu_time, e_fgo, ref_euler_deg, save_dir=f'{REPORT_DIR}/fgo')

Глава 3) Сравниваем (метрики и графики)

In [6]:
from src.analysis.run_methods import run_all_methods
from src.analysis.frames import prepare_scenario

# 3.2
# Сценарии: 01_stationary, 02_acceleration, 03_straight_cruise,
#           04_braking, 05_turn, 06_uphill, 07_downhill_turn,
#           10_accel_cruise_brake, 99_full_mission

S = prepare_scenario('02_acceleration', MOTION_DIR, DATA_DIR)
# S = prepare_scenario('05_turn', MOTION_DIR, DATA_DIR)

# res = run_all_methods(S, use_vibration=False)
res = run_all_methods(S, use_vibration=True)

pos = {m: res[m]['pos'] for m in res}
vel = {m: res[m]['vel'] for m in res}
eul = {m: res[m]['euler'] for m in res}
ref_eul = np.rad2deg(S['ref_euler_ned'])


------------------------------------------------------------
Sample frequency of IMU: [fs] = 400.0 Hz
Reference frame: 0
Simulation time duration: 20.0 s
Simulation runs: 1

------------------------------------------------------------
Simulation results are saved to /home/rsadovec/prj/KAMAZ_Jupyter/generated_data/02_acceleration
The following results are saved:
	time: sample time
	ref_pos: true LLA pos in the navigation frame
	ref_vel: true vel in the NED frame
	ref_att_euler: true attitude (Euler angles, ZYX)
	ref_accel: true accel in the body frame
	ref_gyro: true angular velocity in the body frame
	gps_time: GPS sample time
	ref_gps: true GPS LLA position and NED velocity
	gps_visibility: GPS visibility
	accel: accel measurements
	gyro: gyro measurements
	gps: GPS LLA position and NED velocity measurements
	ref_att_quat: true attitude (quaternion)



In [7]:
from src.analysis.metrics import print_comparison

# Metrics
print_comparison(res, S['ref_pos_ned'], S['ref_vel_ned'], ref_eul)

Метод          RMS|dr|,м   макс|dr|,м   фин|dr|,м
-------------------------------------------------
EKF               0.1625       0.3461      0.3461
UKF               0.1231       0.2141      0.2093
FGO               0.0488       0.9761      0.9761


[{'method': 'EKF',
  'pos_RMS_m': 0.16249674562900426,
  'pos_max_m': 0.3461166365342511,
  'pos_final_m': 0.3461166365342511,
  'vel_RMS_ms': 0.08584971731919006,
  'yaw_RMS_deg': 1.8102009226708609,
  'pitch_RMS_deg': 0.08428778952447818,
  'roll_RMS_deg': 0.12540038834506304},
 {'method': 'UKF',
  'pos_RMS_m': 0.12305728246517747,
  'pos_max_m': 0.21412359405935416,
  'pos_final_m': 0.20928464329203958,
  'vel_RMS_ms': 0.06297213547198956,
  'yaw_RMS_deg': 0.05858516041392089,
  'pitch_RMS_deg': 0.09069969616860357,
  'roll_RMS_deg': 0.10521678744922881},
 {'method': 'FGO',
  'pos_RMS_m': 0.04882430916569129,
  'pos_max_m': 0.9761400302313039,
  'pos_final_m': 0.9761400302313039,
  'vel_RMS_ms': 0.01868768111380325,
  'yaw_RMS_deg': 1.4755010690385653,
  'pitch_RMS_deg': 0.01256578208201352,
  'roll_RMS_deg': 0.08567850858464399}]

In [ ]:
from src.analysis.compare_viz import (
   plot_overlay_trajectory, plot_overlay_error, plot_overlay_attitude, 
   plot_overlay_speed, plot_overlay_yaw_rate, plot_overlay_trajectory_3d,
   plot_overlay_velocity_error
   )

# Trajectory
_ = plot_overlay_trajectory_3d(S['ref_pos_ned'], pos, save_dir=f'{REPORT_DIR}/compare')

# plot_overlay_trajectory(S['ref_pos_ned'], pos, plane='NE', save_dir=f'{REPORT_DIR}/compare')
# plot_overlay_trajectory(S['ref_pos_ned'], pos, plane='ND', save_dir=f'{REPORT_DIR}/compare')
# plot_overlay_trajectory(S['ref_pos_ned'], pos, plane='ED', save_dir=f'{REPORT_DIR}/compare')

# Speed
# _ = plot_overlay_speed(S['imu_time'], S['ref_vel_ned'], vel, save_dir=f'{REPORT_DIR}/compare')

# Attitude
_ = plot_overlay_attitude(S['imu_time'], ref_eul, {m: res[m]['euler'] for m in res}, which='roll', save_dir=f'{REPORT_DIR}/compare')
_ = plot_overlay_attitude(S['imu_time'], ref_eul, {m: res[m]['euler'] for m in res}, which='pitch', save_dir=f'{REPORT_DIR}/compare')
_ = plot_overlay_attitude(S['imu_time'], ref_eul, {m: res[m]['euler'] for m in res}, which='yaw', save_dir=f'{REPORT_DIR}/compare')

# _ = plot_overlay_yaw_rate(S['imu_time'], ref_eul, eul, fs=400.0,
#                       ref_yaw_rate=S['ref_gyro'],     # если добавили в prepare_scenario
#                       save_dir=f'{REPORT_DIR}/ch32')

# Errors
_ = plot_overlay_error(S['imu_time'], S['ref_pos_ned'], pos, logy=False, save_dir=f'{REPORT_DIR}/compare')
_ = plot_overlay_velocity_error(S['imu_time'], S['ref_vel_ned'], vel, logy=False, save_dir=f'{REPORT_DIR}/compare')

In [ ]:
# from src.analysis.compare_viz import plot_clean_vs_vib

# plot_clean_vs_vib(S['imu_time'], S['ref_pos_ned'], res, res_vib, method='EKF', save_dir=f'{REPORT_DIR}/ch33_acc')